# EDA: primer vistazo a la data

Dos tablas:

- `merchant_rating_featureset_sample.parquet`: una fila por merchant y fecha, con features de volumen, aprobación, red, industria y segmentos.
- `txn_sample.parquet`: una fila por transacción (`amount`, tipo de pago, si se aprobó y si es fraude).

El archivo de merchants pesa ~2.3 GB (~10 M filas). Para ver cómo viene la data se carga una muestra; el conteo de filas es el del archivo completo.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

DATA_DIR = next(p for p in (Path("data"), Path("../data")) if p.exists())
MERCHANT_PATH = DATA_DIR / "merchant_rating_featureset_sample.parquet"
TXN_PATH = DATA_DIR / "txn_sample.parquet"

SAMPLE_ROWS = 100_000

In [4]:
def parquet_overview(path):
    pf = pq.ParquetFile(path)
    print(f"{path.name}")
    print(f"  filas: {pf.metadata.num_rows:,}")
    print(f"  columnas: {pf.metadata.num_columns}")
    print(f"  row groups: {pf.metadata.num_row_groups}")
    return pf


merchant_file = parquet_overview(MERCHANT_PATH)
txn_file = parquet_overview(TXN_PATH)

merchant_rating_featureset_sample.parquet
  filas: 10,220,433
  columnas: 71
  row groups: 19
txn_sample.parquet
  filas: 7,079,949
  columnas: 6
  row groups: 2


In [5]:
merchant = next(merchant_file.iter_batches(batch_size=SAMPLE_ROWS)).to_pandas()
txn = next(txn_file.iter_batches(batch_size=SAMPLE_ROWS)).to_pandas()

# Para cargar el archivo entero (ojo con la RAM del de merchants):
# merchant = pd.read_parquet(MERCHANT_PATH)
# txn = pd.read_parquet(TXN_PATH)

print("merchant sample", merchant.shape)
print("txn sample", txn.shape)

merchant sample (100000, 71)
txn sample (100000, 6)


## Merchants: qué features hay

Clave de la tabla: `merchant_id` + `observation_date`. El resto son features del merchant a esa fecha.

In [6]:
merchant.dtypes.to_frame("dtype")

,dtype
observation_date,object
ind_approved_tpv_P95_90d,float64
ind_approved_txn_sum_90d,float64
me_network_avg_tpv_approval_rate,float64
me_network_avg_tpv_chargeback_rate,float64
me_network_avg_tpv_dispute_rate,float64
me_network_size,float64
me_suspended_merchants_in_network,float64
me_suspended_merchants_in_network_share,float64
me_approved_tpv_sum_30d,float64


In [7]:
merchant.head()

,observation_date,ind_approved_tpv_P95_90d,ind_approved_txn_sum_90d,me_network_avg_tpv_approval_rate,me_network_avg_tpv_chargeback_rate,me_network_avg_tpv_dispute_rate,me_network_size,me_suspended_merchants_in_network,me_suspended_merchants_in_network_share,me_approved_tpv_sum_30d,me_approved_txn_sum_15d,me_approved_txn_sum_1d,me_approved_txn_sum_30d,me_approved_txn_sum_7d,me_attempted_ecommerce_txn_sum_7d,me_attempted_openplatform_txn_sum_7d,me_attempted_pocket_txn_sum_7d,me_attempted_recurring_txn_sum_7d,me_attempted_spei_txn_sum_7d,me_attempted_tpv_avg_1d,me_attempted_tpv_avg_30d,me_attempted_tpv_avg_7d,me_attempted_txn_sum_15d,me_attempted_txn_sum_1d,me_attempted_txn_sum_30d,me_attempted_txn_sum_7d,me_tpv_approval_rate_15d,me_tpv_approval_rate_1d,me_tpv_approval_rate_30d,me_tpv_approval_rate_7d,me_txn_approval_rate_15d,me_txn_approval_rate_1d,me_txn_approval_rate_30d,me_txn_approval_rate_7d,me_lifetime_chargeback_tpv,me_lifetime_dispute_tpv,me_lifetime_approved_tpv,me_lifetime_approved_txn,me_lifetime_tpv_approval_rate,me_lifetime_tpv_avg,me_lifetime_txn_approval_rate,me_acquisition_channel,me_acquisition_sub_channel,me_business_acquisition_channel,me_industry_mcc,me_merchant_created_at_timestamp,merchant_tenure_days,ratio_claims_tpv_180d,approved_txns_90d,approved_tpv_90d,industry_mcc,ind_ratio_claims_tpv_180d,p10_approved_txns_90d,p25_approved_txns_90d,p50_approved_txns_90d,p75_approved_txns_90d,p90_approved_txns_90d,p95_approved_txns_90d,p25_approved_tpv_90d,p50_approved_tpv_90d,p75_approved_tpv_90d,p90_approved_tpv_90d,p95_approved_tpv_90d,is_closed_won_ever,is_veto_vip0,vip0_segment,is_veto_config,config_segment,is_portfolio_managed,is_suspended,merchant_id
0,2025-10-23,789.430671,139111.787972,NaN,NaN,NaN,NaN,NaN,NaN,4301.393898,11.080806,0.000000,16.895733,3.033802,0.0,0.0,0.0,0.0,0.0,-1.011121,254.464104,323.490148,10.956732,0.000000,16.611852,2.975856,0.992710,-1.004752,1.006307,0.997958,0.992807,-1.011047,0.986350,0.991950,NaN,NaN,7.377585e+04,308.0,0.941887,239.815217,0.903809,Online,Unassigned,Online,5812.0,2024-08-23 18:34:57,403.0,0.0,117.0,30744.605632,5812.0,0.179866,18.0,50.0,182.0,594.0,1440.0,2393.0,15463.289754,54595.946983,176267.126885,471752.596415,805885.207188,NaN,NaN,NaN,NaN,NaN,NaN,None,M47C03781983B0C4E
1,2025-10-23,4987.158824,29653.661296,0.870204,0.0,0.0,5.0,0.0,0.0,20065.580136,19.005290,4.064267,18.940498,19.032836,0.0,0.0,0.0,0.0,0.0,861.706410,1047.601776,1057.487185,19.183430,3.979181,18.829151,18.841294,0.995878,0.983773,0.999699,1.000769,0.998801,0.992026,0.982907,1.002057,NaN,NaN,5.263501e+06,2679.0,0.864248,2089.733312,0.903360,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,VIP,0.0,NaN,NaN,None,MF83A3175113603FA
2,2025-10-23,3130.652556,28589.864464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.088138e+04,6.0,0.855139,3441.839893,0.865187,Retail,Oxxo,Retail,5651.0,2024-12-20 18:59:31,270.0,0.0,1.0,5358.548840,5651.0,0.159903,10.0,20.0,46.0,121.0,293.0,479.0,12747.303509,36283.183510,96070.806624,235379.312877,392561.647463,NaN,NaN,NaN,NaN,NaN,NaN,None,M10FA358230204F14
3,2025-10-23,4393.856055,3665.478591,NaN,NaN,NaN,NaN,NaN,NaN,6293.678552,62.026700,1.994347,135.861231,26.143991,0.0,0.0,0.0,0.0,0.0,29.380404,45.868468,51.284123,73.390945,1.987556,151.538399,30.227946,0.867323,0.996946,0.904665,0.868777,0.855508,0.985028,0.880354,0.865112,NaN,NaN,1.413139e+04,271.0,0.857412,51.560277,0.894297,Online,ClipMX,Online,5046.0,2025-08-20 04:01:15,42.0,0.0,183.0,10141.319518,5046.0,0.168288,11.0,23.0,77.0,325.0,1048.0,1926.0,7919.330905,25644.914053,78393.899498,213708.224759,363638.580183,NaN,NaN,NaN,NaN,NaN,NaN,None,M5F0D8CBD7871B763
4,2025-10-23,3139.544046,22417.611727,NaN,NaN,NaN,NaN,NaN,NaN,16037.578508,4.018610,0.999570,13.090914,3.031901,0.0,0.0,0.0,0.0,0.0,1201.981682,1233.011999,1178.810493,4.032467,1.023492,12.862572,3.032111,1.017326,1.005649,1.

In [8]:
FEATURE_GROUPS = {
    "id y fecha": [
        "merchant_id",
        "observation_date",
    ],
    "perfil del merchant": [
        "me_acquisition_channel",
        "me_acquisition_sub_channel",
        "me_business_acquisition_channel",
        "me_industry_mcc",
        "industry_mcc",
        "me_merchant_created_at_timestamp",
        "merchant_tenure_days",
    ],
    "segmentos y flags": [
        "is_closed_won_ever",
        "is_veto_vip0",
        "vip0_segment",
        "is_veto_config",
        "config_segment",
        "is_portfolio_managed",
        "is_suspended",
    ],
    "volumen aprobado": [
        c for c in merchant.columns if c.startswith("me_approved_")
    ],
    "volumen intentado": [
        c for c in merchant.columns if c.startswith("me_attempted_")
    ],
    "tasas de aprobación": [
        c for c in merchant.columns if "approval_rate" in c and c.startswith("me_") and not c.startswith("me_lifetime") and not c.startswith("me_network")
    ],
    "lifetime": [
        c for c in merchant.columns if c.startswith("me_lifetime_")
    ],
    "red del merchant": [
        c for c in merchant.columns if "network" in c
    ],
    "benchmarks de industria": [
        c for c in merchant.columns if c.startswith("ind_")
    ],
    "percentiles de industria (90d)": [
        c for c in merchant.columns if c[:1] == "p" and c[1].isdigit()
    ],
    "actividad y claims": [
        "approved_txns_90d",
        "approved_tpv_90d",
        "ratio_claims_tpv_180d",
    ],
}

grouped = {col for cols in FEATURE_GROUPS.values() for col in cols}
missing = [c for c in merchant.columns if c not in grouped]
if missing:
    FEATURE_GROUPS["sin grupo"] = missing

for name, cols in FEATURE_GROUPS.items():
    print(f"\n{name} ({len(cols)})")
    for col in cols:
        print(f"  {col:<45} {merchant[col].dtype}")


id y fecha (2)
  merchant_id                                   str
  observation_date                              object

perfil del merchant (7)
  me_acquisition_channel                        str
  me_acquisition_sub_channel                    str
  me_business_acquisition_channel               str
  me_industry_mcc                               float64
  industry_mcc                                  float64
  me_merchant_created_at_timestamp              datetime64[ns]
  merchant_tenure_days                          float64

segmentos y flags (7)
  is_closed_won_ever                            float64
  is_veto_vip0                                  float64
  vip0_segment                                  str
  is_veto_config                                float64
  config_segment                                str
  is_portfolio_managed                          float64
  is_suspended                                  object

volumen aprobado (5)
  me_approved_tpv_sum_30d            

In [9]:
null_pct = (merchant.isna().mean() * 100).round(1).sort_values(ascending=False)
print("nulos (%)")
print(null_pct.to_string())

nulos (%)
config_segment                             100.0
is_veto_vip0                                99.2
is_veto_config                              99.2
vip0_segment                                99.2
is_portfolio_managed                        98.3
me_lifetime_chargeback_tpv                  98.0
me_lifetime_dispute_tpv                     97.0
is_suspended                                96.8
is_closed_won_ever                          96.8
me_suspended_merchants_in_network_share     75.1
me_suspended_merchants_in_network           75.1
me_network_avg_tpv_approval_rate            75.1
me_network_avg_tpv_chargeback_rate          75.1
me_network_avg_tpv_dispute_rate             75.1
me_network_size                             75.1
me_business_acquisition_channel             29.8
me_industry_mcc                             29.8
me_acquisition_channel                      29.8
me_acquisition_sub_channel                  29.8
me_merchant_created_at_timestamp            29.8
ratio_clai

In [10]:
cat_cols = merchant.select_dtypes(include=["object", "string", "bool", "category"]).columns.tolist()
low_card = [
    c for c in merchant.columns
    if c not in cat_cols and merchant[c].nunique(dropna=False) <= 15
]

for col in cat_cols + low_card:
    print("\n", col)
    print(merchant[col].value_counts(dropna=False).head(12).to_string())


 observation_date
observation_date
2025-10-03    24355
2025-10-23    24045
2025-10-02    24024
2025-10-22    23845
2025-04-10     1879
2025-04-09     1852

 me_acquisition_channel
me_acquisition_channel
Retail                  31260
NaN                     29808
Online                  20393
Resellers               15098
Middle Market            2869
Middle Market MG EXP      423
Software                  102
unknown                    47

 me_acquisition_sub_channel
me_acquisition_sub_channel
NaN             29808
Clip-agentes    12968
ClipMX          12641
Oxxo             5969
Sams             5198
CNP              3706
Office Depot     3345
unknown          3038
Amazon           2760
Unassigned       2732
Coppel           1879
Rutas            1590

 me_business_acquisition_channel
me_business_acquisition_channel
Retail           31260
NaN              29808
Online           20393
Resellers        15098
Middle Market     1728
Rutas             1217
Large Market       347
Software 

In [11]:
num_cols = merchant.select_dtypes(include="number").columns
merchant[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
ind_approved_tpv_P95_90d,90589.0,3.517184e+03,2.448377e+03,366.034828,1070.918825,3240.931104,5002.318069,1.793747e+04
ind_approved_txn_sum_90d,90589.0,1.064313e+06,3.050453e+06,493.944465,11848.276569,29560.729606,140683.222418,1.163732e+07
me_network_avg_tpv_approval_rate,24931.0,8.163897e-01,1.938681e-01,0.000000,0.757472,0.890409,0.944471,1.037092e+00
me_network_avg_tpv_chargeback_rate,24861.0,1.287769e-03,2.248604e-02,0.000000,0.000000,0.000000,0.000000,1.012428e+00
me_network_avg_tpv_dispute_rate,24861.0,1.698044e-03,2.495563e-02,0.000000,0.000000,0.000000,0.000000,1.007315e+00
me_network_size,24937.0,1.650837e+02,1.517353e+03,2.000000,2.000000,2.000000,4.000000,1.528700e+04
me_suspended_merchants_in_network,24937.0,5.795821e+01,5.540092e+02,0.000000,0.000000,0.000000,0.000000,5.585000e+03
me_suspended_merchants_in_network_share,24937.0,5.430814e-02,1.424739e-01,0.000000,0.000000,0.000000,0.000000,9.838756e-01
me_approved_tpv_sum_30d,80070.0,2.765035e+04,2.149477e+05,0.000000,1796.020604,6195.234123,21053.497615,3.792753e+07
me_approved_txn_sum_15d,80070.0,3.348665e+01,1.591759e+02,0.000000,0.999452,3.027819,16.199621,1.682460e+04


## Transacciones

`merchant_id` es la llave para cruzar con la tabla de features. `approved` e `is_fraud` vienen como 0/1.

In [12]:
txn.dtypes.to_frame("dtype")

,dtype
amount,float64
payment_type,str
approved,int32
is_fraud,int32
merchant_id,str
transaction_id,str


In [13]:
txn.head(10)

,amount,payment_type,approved,is_fraud,merchant_id,transaction_id
0,19.874981,CP,1,0,M088E37D2CA114D7C,T27C66C65F497FE0F
1,50.058673,CP,1,0,M931B5C779ECA4F48,TB00AFE0DC6FD5D9B
2,38.089314,CP,1,0,M0B219982B2F2304E,T0912E763F45995B8
3,78.784616,CP,1,0,MF86F4F3B4A99B8A4,T6D44829CF4BEFD32
4,249.909107,CP,1,0,MCF40A9DAFEC3B082,TE7D2DD33CCCA2E72
5,415.977493,CP,1,0,M91F421518C27B79E,T77CDC61B67D23D6A
6,159.721115,CP,1,0,M4A3142EDDAD9E14E,T3EDC9B6A6004E249
7,146.306923,CP,1,0,M54F2F2322C32F221,T71CFDD655B62151C
8,58.605490,CP,1,0,M04FAF1F03FAA9F1E,T1D0801B83D4B060A
9,49.359139,CP,1,0,M34688F6DE9668D17,TA382D689179AF70E


In [14]:
print("nulos (%)")
print((txn.isna().mean() * 100).round(1).to_string())

print("\npayment_type")
print(txn["payment_type"].value_counts(dropna=False).to_string())

print("\napproved")
print(txn["approved"].value_counts(dropna=False).to_string())

print("\nis_fraud")
print(txn["is_fraud"].value_counts(dropna=False).to_string())

print("\namount")
print(txn["amount"].describe().to_string())

nulos (%)
amount            0.0
payment_type      0.0
approved          0.0
is_fraud          0.0
merchant_id       0.0
transaction_id    0.0

payment_type
payment_type
CP     98934
CNP     1066

approved
approved
1    93059
0     6941

is_fraud
is_fraud
0    99950
1       50

amount
count    100000.000000
mean        539.763239
std        2346.713471
min           0.009870
25%          79.873999
50%         178.894196
75%         419.592270
max      334347.237019


## Construcción del target

`is_suspended` describe el estado del merchant en cada fecha, no un evento futuro. Antes de convertirlo en target se mira sobre la tabla completa (solo tres columnas, para no cargar 2.3 GB):

1. Cómo se reparte el estado por mes.
2. Cuántos merchants llegan a estar suspendidos, cuánto dura la suspensión y si vuelven a operar.
3. Cuántos merchants tienen un inicio de suspensión observable (un día activo seguido de uno suspendido).

In [15]:
status = pd.read_parquet(MERCHANT_PATH, columns=["merchant_id", "observation_date", "is_suspended"])
status["observation_date"] = pd.to_datetime(status["observation_date"])
status["estado"] = (
    status["is_suspended"].astype("object")
    .map({True: "suspendido", False: "activo"})
    .fillna("nulo")
)

print(f"filas: {len(status):,}")
print(f"merchants: {status['merchant_id'].nunique():,}")
print(f"fechas: {status['observation_date'].min().date()} a {status['observation_date'].max().date()}")

pd.crosstab(status["observation_date"].dt.to_period("M"), status["estado"], margins=True)

filas: 10,220,433
merchants: 40,265
fechas: 2025-04-01 a 2025-12-31


estado,activo,nulo,suspendido,All
observation_date,,,,
2025-04,861915,130019,113,992047
2025-05,917915,136345,119,1054379
2025-06,917878,133597,79,1051554
2025-07,980116,139393,99,1119608
2025-08,1017933,140498,46,1158477
2025-09,1029861,137563,140,1167564
2025-10,0,1233676,0,1233676
2025-11,0,1198537,0,1198537
2025-12,0,1244591,0,1244591


De octubre a diciembre `is_suspended` viene 100% nulo: esos meses no sirven para entrenar y quedan como período a puntuar. El análisis del target usa solo abril a septiembre.

In [16]:
LABEL_END = pd.Timestamp("2025-09-30")

labelled = (
    status[status["observation_date"] <= LABEL_END]
    .sort_values(["merchant_id", "observation_date"])
    .reset_index(drop=True)
)
labelled["estado_previo"] = labelled.groupby("merchant_id")["estado"].shift()

# Inicio de suspensión: el merchant pasa de activo a suspendido de un día al otro
onsets = labelled.loc[
    (labelled["estado"] == "suspendido") & (labelled["estado_previo"] == "activo"),
    ["merchant_id", "observation_date"],
].rename(columns={"observation_date": "onset_date"})

suspended_days = labelled[labelled["estado"] == "suspendido"].groupby("merchant_id").size()
suspended_ids = suspended_days.index
last_state = labelled[labelled["merchant_id"].isin(suspended_ids)].groupby("merchant_id")["estado"].last()

print(f"filas con etiqueta (abr-sep): {len(labelled):,}")
print(labelled["estado"].value_counts().to_string())
print()
print(f"merchants suspendidos alguna vez: {len(suspended_ids)}")
print(f"  con inicio observado (activo -> suspendido): {onsets['merchant_id'].nunique()}")
print(f"  vuelven a operar al final: {int((last_state == 'activo').sum())}")
print(f"  suspendidos 3 días o menos: {int((suspended_days <= 3).sum())}")
print()
print("días suspendido por merchant")
print(suspended_days.describe().round(1).to_string())

filas con etiqueta (abr-sep): 6,543,629
estado
activo        5725618
nulo           817415
suspendido        596

merchants suspendidos alguna vez: 26
  con inicio observado (activo -> suspendido): 19
  vuelven a operar al final: 19
  suspendidos 3 días o menos: 10

días suspendido por merchant
count     26.0
mean      22.9
std       33.7
min        1.0
25%        2.0
50%        6.5
75%       21.8
max      120.0


In [20]:
import numpy as np


## Calidad de las features sospechosas

Tres cosas que se vieron en la muestra y hay que medir en la tabla completa, porque cambian qué se puede usar como predictor:

- `merchant_tenure_days` a veces baja de un día para el otro.
- Hay tasas de aprobación negativas.
- Hay conteos de transacciones con decimales.

In [21]:
quality_cols = [
    "merchant_id",
    "observation_date",
    "merchant_tenure_days",
    "is_suspended",
]
quality_cols += [c for c in merchant_file.schema_arrow.names if "rate" in c]
quality_cols += [
    c for c in merchant_file.schema_arrow.names
    if "txn_sum" in c or c in ("me_lifetime_approved_txn", "approved_txns_90d")
]

quality = pd.read_parquet(MERCHANT_PATH, columns=quality_cols)
quality["observation_date"] = pd.to_datetime(quality["observation_date"])
print(f"filas: {len(quality):,}  columnas: {quality.shape[1]}")


filas: 10,220,433  columnas: 33


### Antigüedad

Si `merchant_tenure_days` fuera la edad real del merchant, entre dos fechas consecutivas debería subir 1. Se mira solo los pares de filas del mismo merchant separadas por exactamente un día.

In [22]:
tenure = quality[["merchant_id", "observation_date", "merchant_tenure_days"]].sort_values(
    ["merchant_id", "observation_date"]
)
by_merchant = tenure.groupby("merchant_id", sort=False)
tenure["gap_dias"] = by_merchant["observation_date"].diff().dt.days
tenure["delta_tenure"] = by_merchant["merchant_tenure_days"].diff()

consec = tenure.loc[tenure["gap_dias"].eq(1) & tenure["delta_tenure"].notna(), "delta_tenure"]
merchants_con_tenure = tenure.loc[tenure["merchant_tenure_days"].notna(), "merchant_id"].nunique()
merchants_con_baja = tenure.loc[
    tenure["gap_dias"].eq(1) & tenure["delta_tenure"].lt(-0.5), "merchant_id"
].nunique()

print(f"pares de días consecutivos: {len(consec):,}")
print(f"baja (delta < -0.5): {(consec < -0.5).mean():.1%}")
print(f"se queda ( |delta| <= 0.5): {(consec.abs() <= 0.5).mean():.1%}")
print(f"sube (delta > 0.5): {(consec > 0.5).mean():.1%}")
print(f"merchants con al menos una baja: {merchants_con_baja:,} de {merchants_con_tenure:,}")
print()
print(consec.describe().round(2).to_string())


pares de días consecutivos: 7,388,518
baja (delta < -0.5): 43.9%
se queda ( |delta| <= 0.5): 10.0%
sube (delta > 0.5): 46.1%
merchants con al menos una baja: 37,598 de 37,948

count    7388518.00
mean           0.77
std           19.58
min         -238.00
25%           -6.00
50%            0.00
75%            7.00
max          228.00


En días consecutivos la antigüedad **baja el 44% de las veces** y sube el 46%. Casi todos los merchants (37.598 de 37.948) tienen al menos una baja. La media del cambio es +0,8 días, cerca de +1, con un desvío de ~20 días.

Lectura: la tendencia de fondo es la edad real (+1 por día), pero cada foto trae ruido grande. `merchant_tenure_days` no sirve como número exacto. Como máximo, como orden de magnitud de la antigüedad.

### Tasas

Una tasa real vive en [0, 1]. Abajo, el porcentaje se calcula sobre las filas no nulas.

In [23]:
rate_cols = [c for c in quality.columns if "rate" in c]

rate_rows = []
for col in rate_cols:
    values = quality[col].dropna()
    rate_rows.append({
        "feature": col,
        "nulos_%": round(quality[col].isna().mean() * 100, 1),
        "neg_%": round((values < 0).mean() * 100, 2),
        "mayor_a_1_%": round((values > 1).mean() * 100, 2),
        "min": round(values.min(), 3),
        "p50": round(values.median(), 3),
        "max": round(values.max(), 3),
    })

rate_quality = pd.DataFrame(rate_rows).sort_values("neg_%", ascending=False)
rate_quality


,feature,nulos_%,neg_%,mayor_a_1_%,min,p50,max
4,me_tpv_approval_rate_1d,21.9,67.83,12.78,-1.054,-0.994,1.050
8,me_txn_approval_rate_1d,21.9,67.83,12.71,-1.054,-0.994,1.047
6,me_tpv_approval_rate_7d,21.9,31.96,22.21,-1.050,0.954,1.054
10,me_txn_approval_rate_7d,21.9,31.96,21.92,-1.050,0.946,1.050
3,me_tpv_approval_rate_15d,21.9,15.32,24.87,-1.047,0.982,1.049
7,me_txn_approval_rate_15d,21.9,15.32,24.45,-1.049,0.979,1.049
1,me_network_avg_tpv_chargeback_rate,75.0,0.00,0.01,0.000,0.000,1.034
0,me_network_avg_tpv_approval_rate,74.9,0.00,2.96,0.000,0.891,1.040
2,me_network_avg_tpv_dispute_rate,75.0,0.00,0.01,0.000,0.000,1.031
5,me_tpv_approval_rate_30d,21.9,0.00,26.06,0.000,0.986,1.054


El ruido es más fuerte cuanto más corta es la ventana:

| Ventana | Negativas | Mayores a 1 | Mediana |
|---|---:|---:|---:|
| 1 día | 68% | 13% | −0,99 |
| 7 días | 32% | 22% | 0,95 |
| 15 días | 15% | 25% | 0,98 |
| 30 días, lifetime y red | 0% | 3% a 26% | ~0,9 |

La tasa de 1 día está dominada por el ruido: la mediana es negativa. Las de 30 días, lifetime y red no bajan de 0, pero sí se pasan de 1. Antes de usarlas como predictor hay que recortarlas a [0, 1], y la de 1 día es candidata a salir.

### Conteos

`me_lifetime_approved_txn` y `approved_txns_90d` son enteros. El resto de los `*_txn_sum` se mide como porcentaje de valores no enteros, otra vez sobre filas no nulas.

In [24]:
count_cols = [
    c for c in quality.columns
    if "txn_sum" in c or c in ("me_lifetime_approved_txn", "approved_txns_90d")
]

count_rows = []
for col in count_cols:
    values = quality[col].dropna()
    frac = (values % 1).abs()
    count_rows.append({
        "feature": col,
        "nulos_%": round(quality[col].isna().mean() * 100, 1),
        "no_entero_%": round((frac > 1e-6).mean() * 100, 2),
        "min": round(values.min(), 3),
        "max": round(values.max(), 1),
    })

count_quality = pd.DataFrame(count_rows).sort_values("no_entero_%", ascending=False)
count_quality


,feature,nulos_%,no_entero_%,min,max
0,ind_approved_txn_sum_90d,9.1,100.00,447.768,13804046.7
12,me_attempted_txn_sum_30d,21.9,100.00,0.953,44545.3
3,me_approved_txn_sum_30d,21.9,98.53,0.000,39218.5
10,me_attempted_txn_sum_15d,21.9,84.67,0.000,25143.8
1,me_approved_txn_sum_15d,21.9,83.35,0.000,21704.7
13,me_attempted_txn_sum_7d,21.9,68.04,0.000,12862.6
4,me_approved_txn_sum_7d,21.9,66.97,0.000,10790.3
11,me_attempted_txn_sum_1d,21.9,32.17,0.000,2556.8
2,me_approved_txn_sum_1d,21.9,31.69,0.000,2012.8
5,me_attempted_ecommerce_txn_sum_7d,21.9,5.36,0.000,1723.3


Los conteos de ventanas cortas y `ind_approved_txn_sum_90d` vienen con decimales (la de 30 días, en el 99% de las filas). `me_lifetime_approved_txn` y `approved_txns_90d` son enteros limpios. `me_attempted_pocket_txn_sum_7d` es siempre 0.

Mismo patrón que en las tasas: a las sumas por ventana les agregaron ruido. Sirven como volumen aproximado, no como conteo exacto.

## Transacciones de merchants suspendidos vs el resto

La tabla de transacciones no tiene fecha, así que este cruce es descriptivo: no se puede usar `is_fraud` como feature de una foto del merchant sin mezclar futuro y pasado. Sirve para ver si la suspensión y el fraude van juntos.

In [ ]:
txn_full = pd.read_parquet(TXN_PATH, columns=["merchant_id", "approved", "is_fraud", "amount"])
suspended_ids = set(quality.loc[quality["is_suspended"].eq(True), "merchant_id"])

txn_full["grupo"] = np.where(
    txn_full["merchant_id"].isin(suspended_ids),
    "suspendido alguna vez",
    "nunca suspendido",
)

txn_by_group = txn_full.groupby("grupo").agg(
    txns=("approved", "size"),
    merchants=("merchant_id", "nunique"),
    tasa_aprobacion=("approved", "mean"),
    tasa_fraude=("is_fraud", "mean"),
    txns_fraude=("is_fraud", "sum"),
    monto_p50=("amount", "median"),
)
txn_by_group["share_txns"] = txn_by_group["txns"] / txn_by_group["txns"].sum()
txn_by_group["share_fraude"] = txn_by_group["txns_fraude"] / txn_by_group["txns_fraude"].sum()
txn_by_group.round(4)


,txns,merchants,tasa_aprobacion,tasa_fraude,txns_fraude,monto_p50,share_txns,share_fraude
grupo,,,,,,,,
nunca suspendido,7076098,40239,0.9345,0.0003,2342,146.7767,0.9995,0.9008
suspendido alguna vez,3851,26,0.7863,0.0670,258,1141.4658,0.0005,0.0992


: 

Los 26 merchants suspendidos están todos en la tabla de transacciones.

| | Nunca suspendido | Suspendido alguna vez |
|---|---:|---:|
| Transacciones | 7.076.098 | 3.851 (0,05%) |
| Tasa de aprobación | 93,4% | 78,6% |
| Tasa de fraude | 0,033% | 6,7% |
| Monto mediano | 147 | 1.141 |
| Share del fraude total | 90,1% | **9,9%** |

Con el 0,05% de las transacciones concentran el 10% del fraude, aprueban peor y mueven montos más altos. La suspensión marca merchants distintos, no un flag vacío. Igual, sin fecha en la transacción, `is_fraud` queda para medir el impacto de las reglas, no para entrenar.